<a href="https://colab.research.google.com/github/SharkTechLLC/businesstaxprime-website/blob/main/update_nvc_timeframes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NVC Timeframes Gist Updater

Use this notebook when the home scheduler cannot fetch `travel.state.gov` directly.

Workflow:

1. Open the official NVC Timeframes page in a browser.
2. Save or copy the page HTML.
3. Run this notebook.
4. Upload the saved HTML file or paste the HTML text.
5. Review the parsed JSON.
6. Publish `nvc_timeframes.json` to the Visa Bulletin Gist.

Do not hardcode tokens in this notebook. Use Colab Secrets or the secure prompts below.

In [1]:
import json
import os
import re
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Any

import requests

try:
    from google.colab import files, userdata
    IN_COLAB = True
except Exception:
    files = None
    userdata = None
    IN_COLAB = False

OFFICIAL_URL = "https://travel.state.gov/content/travel/en/us-visas/immigrate/nvc-timeframes.html"
GIST_FILENAME = "nvc_timeframes.json"
GIST_DESCRIPTION = "Visa Bulletin JSON feed"

DATE_RE = re.compile(r"(\d{1,2})-([A-Za-z]{3})-(\d{2,4})")
MONTHS = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4,
    "may": 5, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}
SECTION_CONFIGS = {
    "case_creation": {
        "anchors": [
            r"Current case creation time frame",
            r"Current case file creation time",
            r"case creation time frame",
            r"case file creation time",
        ],
        "description": r"working on cases that were received from USCIS on",
    },
    "case_review": {
        "anchors": [r"Current case review time", r"case review time"],
        "description": r"reviewing documents submitted to us on",
    },
    "public_inquiry": {
        "anchors": [
            r"Current Public Inquiry Form response time",
            r"Public Inquiry Form response time",
            r"public inquiry response time",
        ],
        "description": r"responding to inquiries received on",
    },
}

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def html_to_text(html: str) -> str:
    without_scripts = re.sub(r"<(script|style)\b[^>]*>.*?</\\1>", " ", html, flags=re.IGNORECASE | re.DOTALL)
    without_tags = re.sub(r"<[^>]+>", " ", without_scripts)
    return re.sub(r"\s+", " ", without_tags).strip()

def parse_state_dept_date(raw: str) -> str:
    match = DATE_RE.search(raw.strip())
    if not match:
        raise ValueError(f"Could not parse NVC date: {raw}")
    month = MONTHS.get(match.group(2).lower())
    if not month:
        raise ValueError(f"Could not parse NVC month: {raw}")
    year_text = match.group(3)
    year = 2000 + int(year_text) if len(year_text) == 2 else int(year_text)
    return f"{year:04d}-{month:02d}-{int(match.group(1)):02d}"

def date_from_groups(match: re.Match[str], start: int) -> str:
    return parse_state_dept_date("-".join(match.group(i) for i in range(start, start + 3)))

def sentence_containing(text: str, pattern: str) -> str | None:
    sentence_match = re.search(r"[^.]*" + pattern + r"[^.]*\.", text, re.IGNORECASE)
    if not sentence_match:
        return None
    return sentence_match.group(0).strip()

def parse_section(text: str, key: str) -> tuple[str, str, str]:
    config = SECTION_CONFIGS[key]
    description = config["description"]
    exact = re.search(
        r"As of\s+(%s),\s+we are\s+%s\s+(%s)\." % (DATE_RE.pattern, description, DATE_RE.pattern),
        text,
        re.IGNORECASE,
    )
    if exact:
        sentence = sentence_containing(text, re.escape(exact.group(0))) or exact.group(0)
        return date_from_groups(exact, 1), date_from_groups(exact, 4), sentence

    for anchor in config["anchors"]:
        anchored = re.search(
            anchor + r".{0,500}?As of\s+(%s).{0,300}?%s\s+(%s)\." % (DATE_RE.pattern, description, DATE_RE.pattern),
            text,
            re.IGNORECASE,
        )
        if anchored:
            sentence = sentence_containing(text, description) or anchored.group(0).strip()
            return date_from_groups(anchored, 1), date_from_groups(anchored, 4), sentence

    for anchor in config["anchors"]:
        heading = re.search(anchor, text, re.IGNORECASE)
        if not heading:
            continue
        fragment = text[heading.start() : heading.start() + 700]
        dates = DATE_RE.findall(fragment)
        if len(dates) >= 2:
            as_of = parse_state_dept_date("-".join(dates[0]))
            timeframe = parse_state_dept_date("-".join(dates[1]))
            sentence = sentence_containing(fragment, DATE_RE.pattern) or fragment[:300].strip()
            return as_of, timeframe, sentence

    raise ValueError(f"Could not parse NVC timeframe section: {key}")

def build_payload(html: str) -> dict[str, Any]:
    text = html_to_text(html)
    items: dict[str, dict[str, str]] = {}
    as_of_dates: set[str] = set()
    for key in SECTION_CONFIGS:
        as_of, timeframe_date, source_sentence = parse_section(text, key)
        as_of_dates.add(as_of)
        items[key] = {"date": timeframe_date, "source_sentence": source_sentence}
    if len(as_of_dates) != 1:
        raise ValueError(f"NVC sections have mismatched as-of dates: {sorted(as_of_dates)}")
    return {
        "schema_version": 1,
        "as_of_date": next(iter(as_of_dates)),
        "updated_at": utc_now(),
        "official_url": OFFICIAL_URL,
        "timeframes": items,
    }

def get_secret(name: str, prompt: str, password: bool = True) -> str:
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if value:
        return value
    return getpass(prompt) if password else input(prompt).strip()

def publish_to_gist(payload: dict[str, Any], gist_id: str, token: str) -> dict[str, Any]:
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
    }
    gist_payload = {
        "description": GIST_DESCRIPTION,
        "files": {
            GIST_FILENAME: {
                "content": json.dumps(payload, indent=2, ensure_ascii=False) + "\n"
            }
        },
    }
    response = requests.patch(f"https://api.github.com/gists/{gist_id}", headers=headers, json=gist_payload, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"Gist update failed: {response.status_code} {response.text}")
    return response.json()

print("Setup loaded.")

Setup loaded.


## Load HTML

Choose one method below. Upload is usually easiest.

In [2]:
# Method A: upload a browser-saved NVC HTML file.
if not IN_COLAB:
    raise RuntimeError("This upload cell is intended for Google Colab.")

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

name, content = next(iter(uploaded.items()))
html = content.decode("utf-8", errors="replace")
print(f"Loaded {name}: {len(html):,} characters")

Saving NVC Timeframes.html to NVC Timeframes.html
Loaded NVC Timeframes.html: 118,497 characters


In [3]:
# Method B: paste HTML manually if upload is inconvenient.
# Run this cell only if you did not use the upload cell above.
# html = input("Paste the full saved NVC HTML here: ")

## Parse And Preview JSON

In [4]:
payload = build_payload(html)
print(json.dumps(payload, indent=2, ensure_ascii=False))

{
  "schema_version": 1,
  "as_of_date": "2026-08-17",
  "updated_at": "2026-08-18T18:52:06Z",
  "official_url": "https://travel.state.gov/content/travel/en/us-visas/immigrate/nvc-timeframes.html",
  "timeframes": {
    "case_creation": {
      "date": "2026-07-23",
      "source_sentence": "Current case file creation time: Current case creation time frame: As of 17-Aug-26, we are working on cases that were received from USCIS on 23-Jul-26. Once USCIS sends your I-797 approval notice, they will send your approved petition to the National Visa Center (NVC) for processing."
    },
    "case_review": {
      "date": "2026-07-01",
      "source_sentence": "Current case review time: Current case review time: As of 17-Aug-26, we are reviewing documents submitted to us on 1-Jul-26. Before the National Visa Center reviews your case, you must pay all fees and submit all required documents, such as: Petitioner’s Affidavit of Support; Supporting financial documents; Applicant’s DS-260, and Applic

## Publish To Gist

This cell updates only the `nvc_timeframes.json` file in the configured Gist.

In [5]:
GITHUB_GIST_TOKEN = get_secret("GITHUB_GIST_TOKEN", "GitHub token: ", password=True)
BULLETIN_GIST_ID = get_secret("BULLETIN_GIST_ID", "Bulletin Gist ID: ", password=False)

answer = input(f"Publish {GIST_FILENAME} to Gist {BULLETIN_GIST_ID}? Type YES: ").strip()
if answer != "YES":
    raise RuntimeError("Publish cancelled.")

result = publish_to_gist(payload, BULLETIN_GIST_ID, GITHUB_GIST_TOKEN)
raw_url = result.get("files", {}).get(GIST_FILENAME, {}).get("raw_url")
print("NVC Gist update successful.")
print("Gist ID:", result.get("id"))
print("Raw URL:", raw_url)

Publish nvc_timeframes.json to Gist ef0a0daae4e6174265cf17fa09340c4f? Type YES: YES
NVC Gist update successful.
Gist ID: ef0a0daae4e6174265cf17fa09340c4f
Raw URL: https://gist.githubusercontent.com/gosharktech/ef0a0daae4e6174265cf17fa09340c4f/raw/4a8c53d9b955834b96a80c2a349db7a11c8f3333/nvc_timeframes.json
